# Unified detector — zero-shot OOD on HC3 (GPU)

Runs the **served unified fake-detector's two DL members** (BiLSTM + DistilBERT) **zero-shot** on
the full HC3 set (8,000 balanced: ChatGPT vs human) — a corpus the detector was **never trained
on** (different generator AND text type). On GPU. The 8 sklearn + all metrics are computed
**locally** (`merge_unified_hc3.py`). This is the defensible OOD generalization number for the
deployed detector.

**Upload to Drive root (`MyDrive/`):**
1. `data/ood_hc3_full.csv`
2. the folder `data/combined_fake_lstm_onnx/`  (combined_fake_lstm.onnx + tokenizer.json + threshold.json)
3. the folder `data/combined_fake_distilbert_onnx/`  (model.onnx + threshold.json + HF tokenizer files)

**Output:** `unified_hc3_dl_preds.csv` (row-aligned with ood_hc3_full.csv) → put it in the project
root, then run `python merge_unified_hc3.py`. Runtime → GPU.

In [ ]:
!pip install -q onnxruntime-gpu transformers

In [ ]:
import os, json, numpy as np, pandas as pd
from google.colab import drive, files
drive.mount('/content/drive')
CSV='/content/drive/MyDrive/ood_hc3_full.csv'
LSTM_DIR='/content/drive/MyDrive/combined_fake_lstm_onnx'
DB_DIR='/content/drive/MyDrive/combined_fake_distilbert_onnx'
LSTM_MAX_LEN, DB_MAX_LEN = 250, 256

df=pd.read_csv(CSV); df['review_text']=df['review_text'].astype(str)
texts=df['review_text'].tolist()
print('rows', len(df), '| AI', int((df.ground_truth_label==1).sum()), '/ human', int((df.ground_truth_label==0).sum()))

In [ ]:
import onnxruntime as ort
from transformers import AutoTokenizer
PROV=['CUDAExecutionProvider','CPUExecutionProvider']
print('providers:', ort.get_available_providers())

class KerasTok:
    def __init__(self, path):
        cfg=json.load(open(path,encoding='utf-8'))['config']
        wi=cfg['word_index']; wi=json.loads(wi) if isinstance(wi,str) else wi
        self.wi=wi; self.lower=cfg.get('lower',True); self.num=cfg.get('num_words')
        self.filt=str.maketrans('','',cfg.get('filters',''))
    def seqs(self, texts):
        out=[]
        for t in texts:
            if self.lower: t=t.lower()
            t=t.translate(self.filt); s=[]
            for w in t.split():
                i=self.wi.get(w)
                if i is not None and (self.num is None or i<self.num): s.append(i)
            out.append(s)
        return out

def pad(seqs, ml):
    a=np.zeros((len(seqs),ml),dtype=np.float32)
    for i,s in enumerate(seqs):
        s=s[:ml]; a[i,:len(s)]=s
    return a

def thr_of(p):
    return float(json.load(open(p)).get('threshold',0.5)) if os.path.exists(p) else 0.5

# unified LSTM (text-only, single input, sigmoid output) — same filenames as model_unified expects
ls=ort.InferenceSession(os.path.join(LSTM_DIR,'combined_fake_lstm.onnx'),providers=PROV)
ltok=KerasTok(os.path.join(LSTM_DIR,'combined_fake_lstm_tokenizer.json'))
lthr=thr_of(os.path.join(LSTM_DIR,'combined_fake_lstm_threshold.json'))
lin=ls.get_inputs()[0].name
def lstm_prob(chunk):
    return ls.run(None,{lin:pad(ltok.seqs(chunk),LSTM_MAX_LEN)})[0].reshape(-1)

# unified DistilBERT (input_ids/attention_mask, 2-logit softmax)
db=ort.InferenceSession(os.path.join(DB_DIR,'model.onnx'),providers=PROV)
dtok=AutoTokenizer.from_pretrained(DB_DIR)
dthr=thr_of(os.path.join(DB_DIR,'threshold.json'))
def db_prob(chunk):
    enc=dtok(chunk,truncation=True,padding='max_length',max_length=DB_MAX_LEN,return_tensors='np')
    feed={x.name:enc[x.name].astype('int64') for x in db.get_inputs() if x.name in enc}
    lg=np.asarray(db.run(None,feed)[0],dtype=np.float64)
    e=np.exp(lg-lg.max(axis=1,keepdims=True))
    return (e/e.sum(axis=1,keepdims=True))[:,1]
print('LSTM thr', lthr, '| DB thr', dthr)

In [ ]:
N=len(texts); lp=[]; dp=[]
for s in range(0,N,4096): lp.append(lstm_prob(texts[s:s+4096]))
for s in range(0,N,256):  dp.append(db_prob(texts[s:s+256]))
lp=np.concatenate(lp); dp=np.concatenate(dp)
out=pd.DataFrame({'review_id':df['review_id'].values,'ground_truth_label':df['ground_truth_label'].values,
                  'lstm_prob':lp,'lstm_pred':(lp>=lthr).astype(int),
                  'db_prob':dp,'db_pred':(dp>=dthr).astype(int)})
out.to_csv('unified_hc3_dl_preds.csv',index=False)
print('saved unified_hc3_dl_preds.csv', out.shape)
print('LSTM AI-pred rate', round(float(out.lstm_pred.mean()),3), '| DB AI-pred rate', round(float(out.db_pred.mean()),3))
files.download('unified_hc3_dl_preds.csv')